In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS gold;

In [0]:
%sql
CREATE OR REPLACE VIEW gold.dim_date AS
WITH all_dates AS (
    SELECT date_received        AS d FROM silver.customer_complaints WHERE date_received IS NOT NULL
    UNION
    SELECT date_sent_to_company AS d FROM silver.customer_complaints WHERE date_sent_to_company IS NOT NULL
)
SELECT
    CAST(date_format(d, 'yyyyMMdd') AS INT) AS date_key,
    d                                       AS full_date,
    year(d)                                 AS year,
    quarter(d)                              AS quarter,
    month(d)                                AS month,
    date_format(d, 'MMMM')                  AS month_name
FROM all_dates;

In [0]:
%sql
CREATE OR REPLACE VIEW gold.dim_product AS
SELECT DISTINCT
    xxhash64(product, sub_product) AS product_key,
    product,
    sub_product
FROM silver.customer_complaints;

In [0]:
%sql
CREATE OR REPLACE VIEW gold.dim_company AS
SELECT DISTINCT
    xxhash64(company) AS company_key,
    company
FROM silver.customer_complaints;

In [0]:
%sql
CREATE OR REPLACE VIEW gold.dim_issue AS
SELECT DISTINCT
    xxhash64(issue, sub_issue) AS issue_key,
    issue,
    sub_issue
FROM silver.customer_complaints;

In [0]:
%sql
CREATE OR REPLACE VIEW gold.dim_geography AS
SELECT DISTINCT
    xxhash64(state, zip3) AS geography_key,
    state,
    zip3
FROM silver.customer_complaints;

In [0]:
%sql
CREATE OR REPLACE VIEW gold.dim_submission AS
SELECT DISTINCT
    xxhash64(submitted_via, consumer_consent_provided, tags) AS submission_key,
    submitted_via,
    consumer_consent_provided,
    tags
FROM silver.customer_complaints;

In [0]:
%sql
CREATE OR REPLACE VIEW gold.dim_response AS
SELECT DISTINCT
    xxhash64(company_response_to_consumer, company_public_response) AS response_key,
    company_response_to_consumer,
    company_public_response
FROM silver.customer_complaints;

In [0]:
%sql
CREATE OR REPLACE VIEW gold.fact_complaints AS
SELECT
    -- degenerate dimension (natural key)
    complaint_id,

    -- foreign keys
    CAST(date_format(date_received,        'yyyyMMdd') AS INT)        AS date_received_key,
    CAST(date_format(date_sent_to_company, 'yyyyMMdd') AS INT)        AS date_sent_key,
    xxhash64(product, sub_product)                                    AS product_key,
    xxhash64(issue, sub_issue)                                        AS issue_key,
    xxhash64(company)                                                 AS company_key,
    xxhash64(state, zip3)                                             AS geography_key,
    xxhash64(submitted_via, consumer_consent_provided, tags)          AS submission_key,
    xxhash64(company_response_to_consumer, company_public_response)   AS response_key,

    -- measures
    days_to_send,
    is_timely_response,
    is_in_progress,
    has_narrative,
    narrative_length,

    -- audit / lineage
    ingestion_timestamp,
    source_system
FROM silver.customer_complaints;

In [0]:
#validation of gold layer
df_silver = spark.table("silver.customer_complaints")
df_fact   = spark.table("gold.fact_complaints")

assert df_fact.count() == df_silver.count(), \
    f"Row count mismatch! Silver has {df_silver.count()} rows but fact_complaints has {df_fact.count()} rows."

print("Gold Layer Validated Successfully")
print(f"Verified Row Count: {df_fact.count()} rows in fact_complaints.")

# Business questions:


In [0]:
%sql
-- Which companies get the most complaints?
SELECT c.company, COUNT(*) AS total_complaints
FROM gold.fact_complaints f
JOIN gold.dim_company c ON f.company_key = c.company_key
GROUP BY c.company
ORDER BY total_complaints DESC
LIMIT 10;

In [0]:
%sql
-- What products and sub-products do people complain about most?
SELECT p.product, p.sub_product, COUNT(*) AS total_complaints
FROM gold.fact_complaints f
JOIN gold.dim_product p ON f.product_key = p.product_key
GROUP BY p.product, p.sub_product
ORDER BY total_complaints DESC
LIMIT 10;

In [0]:
%sql
-- Which companies respond on time and which are slow?
SELECT c.company,
       COUNT(*) AS total_complaints,
       ROUND(AVG(f.days_to_send), 2) AS avg_days_to_send,
       ROUND(SUM(f.is_timely_response) * 100.0 / COUNT(*), 2) AS pct_timely_response
FROM gold.fact_complaints f
JOIN gold.dim_company c ON f.company_key = c.company_key
GROUP BY c.company
HAVING COUNT(*) > 500          -- only companies with meaningful volume
ORDER BY pct_timely_response ASC   -- worst performers first
LIMIT 10;

In [0]:
%sql
-- Which states have the most complaints?
SELECT g.state, COUNT(*) AS total_complaints
FROM gold.fact_complaints f
JOIN gold.dim_geography g ON f.geography_key = g.geography_key
GROUP BY g.state
ORDER BY total_complaints DESC
LIMIT 10;

In [0]:
%sql
-- What specific issues and sub-issues come up most?
SELECT i.issue, i.sub_issue, COUNT(*) AS total_complaints
FROM gold.fact_complaints f
JOIN gold.dim_issue i ON f.issue_key = i.issue_key
GROUP BY i.issue, i.sub_issue
ORDER BY total_complaints DESC
LIMIT 10;

In [0]:
%sql
-- How do companies resolve complaints?
SELECT r.company_response_to_consumer, r.company_public_response, COUNT(*) AS total_complaints
FROM gold.fact_complaints f
JOIN gold.dim_response r ON f.response_key = r.response_key
GROUP BY r.company_response_to_consumer, r.company_public_response
ORDER BY total_complaints DESC;

In [0]:
%sql
-- How are complaints submitted, and how many are still open?
SELECT 
    s.submitted_via,
    s.consumer_consent_provided,
    s.tags,
    COUNT(*) AS total_complaints,
    SUM(f.is_in_progress) AS still_open
FROM gold.fact_complaints f
JOIN gold.dim_submission s ON f.submission_key = s.submission_key
GROUP BY s.submitted_via, s.consumer_consent_provided, s.tags
ORDER BY total_complaints DESC;

In [0]:
%sql
-- What % of complaints include a consumer narrative, by product?
SELECT p.product,
       COUNT(*) AS total_complaints,
       SUM(f.has_narrative) AS with_narrative,
       ROUND(SUM(f.has_narrative) * 100.0 / COUNT(*), 2) AS pct_with_narrative,
       ROUND(AVG(CASE WHEN f.narrative_length IS NOT NULL THEN f.narrative_length END), 0) AS avg_narrative_length
FROM gold.fact_complaints f
JOIN gold.dim_product p ON f.product_key = p.product_key
GROUP BY p.product
ORDER BY pct_with_narrative DESC;

In [0]:
%sql
-- Which companies have the most unresolved (in-progress) complaints?
SELECT c.company,
       COUNT(*)                  AS total_complaints,
       SUM(f.is_in_progress)     AS still_open,
       ROUND(SUM(f.is_in_progress) * 100.0 / COUNT(*), 2) AS pct_open
FROM gold.fact_complaints f
JOIN gold.dim_company c ON f.company_key = c.company_key
GROUP BY c.company
ORDER BY still_open DESC
LIMIT 10;

In [0]:
%sql
-- Which states have the highest % of open complaints?
SELECT g.state,
       COUNT(*)                  AS total_complaints,
       SUM(f.is_in_progress)     AS still_open,
       ROUND(SUM(f.is_in_progress) * 100.0 / COUNT(*), 2) AS pct_open
FROM gold.fact_complaints f
JOIN gold.dim_geography g ON f.geography_key = g.geography_key
GROUP BY g.state
ORDER BY pct_open DESC
LIMIT 10;

In [0]:
%sql
-- Which companies have the lowest rate of consumer-friendly resolutions?
SELECT c.company,
       COUNT(*) AS total_complaints,
       ROUND(SUM(CASE WHEN r.company_response_to_consumer 
             IN ('Closed with explanation', 'Closed with monetary relief', 
                 'Closed with non-monetary relief') 
             THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_resolved_favorably
FROM gold.fact_complaints f
JOIN gold.dim_company  c ON f.company_key  = c.company_key
JOIN gold.dim_response r ON f.response_key = r.response_key
GROUP BY c.company
HAVING COUNT(*) > 500
ORDER BY pct_resolved_favorably ASC
LIMIT 10;

In [0]:
%sql
-- Which product is complained about most in each state?
SELECT g.state, p.product, COUNT(*) AS total_complaints,
       RANK() OVER (PARTITION BY g.state ORDER BY COUNT(*) DESC) AS rnk
FROM gold.fact_complaints f
JOIN gold.dim_geography g ON f.geography_key = g.geography_key
JOIN gold.dim_product   p ON f.product_key   = p.product_key
GROUP BY g.state, p.product
QUALIFY rnk = 1
ORDER BY total_complaints DESC;

In [0]:
%sql
-- Since credit reporting dominates, what is the #2 product per state?
SELECT g.state, p.product, COUNT(*) AS total_complaints,
       RANK() OVER (PARTITION BY g.state ORDER BY COUNT(*) DESC) AS rnk
FROM gold.fact_complaints f
JOIN gold.dim_geography g ON f.geography_key = g.geography_key
JOIN gold.dim_product   p ON f.product_key   = p.product_key
GROUP BY g.state, p.product
QUALIFY rnk = 2
ORDER BY total_complaints DESC;

In [0]:
%sql
-- Does the submission channel affect how quickly companies respond?
SELECT s.submitted_via,
       COUNT(*) AS total_complaints,
       ROUND(AVG(f.days_to_send), 2) AS avg_days_to_send,
       ROUND(SUM(f.is_timely_response) * 100.0 / COUNT(*), 2) AS pct_timely
FROM gold.fact_complaints f
JOIN gold.dim_submission s ON f.submission_key = s.submission_key
GROUP BY s.submitted_via
ORDER BY avg_days_to_send DESC;